# Working with Hydro-PE data
## Zarr version using R

<img width="309" height="57" alt="UKCEH and FDRI logos" src="https://github.com/user-attachments/assets/04afdc63-663f-41e4-b29d-9419f78d76c3" />
</br>

**Authors:** [Matt Dalle Piagge](https://mattjbr123.github.io/) and Kit Macleod. With help from: Matt Fry, Mike Brown, Anna Rose Klaptocz, Faiza Samreen, Matt Coole

---

**Launch this notebook**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NERC-CEH/fdri-gridded-notebooks/blob/main/notebooks/Hydro-PE/hydrope_zarr_R.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/NERC-CEH/fdri-gridded-notebooks/HEAD?labpath=notebooks/Hydro-PE/hydrope_zarr_R.ipynb)

(Ctrl + Click to open in a new tab)

> **Google Colab:** If running in Google Colab, you will need to switch to an R runtime. Click the "Runtime" menu in the menu bar at the top of the screen, select Change runtime type, select R from the 'Runtime type' drop down box, you can leave the rest the same.

Full instructions available in the [accompanying README](https://github.com/NERC-CEH/fdri-gridded-notebooks/blob/main/README.md#R).

---

## About this notebook

This notebook explores the **Hydro-PE** dataset. [Hydro-PE](https://doi.org/10.5285/2aa2c8ab-9e32-4b3b-9636-503912305aca) is an observations-derived gridded Potential Evaporation dataset for the UK produced by [UKCEH](https://www.ceh.ac.uk/) using the [HadUK-Grid](https://dx.doi.org/10.5285/f02cc6ddd92f45b18b9ab6ab544df7d9) dataset. It contains two variables, Potential Evapotranspiration (PE or PET) and Potential Evapotranspiration with Interception correction (PETI). See the supporting documentation on the [catalogue page](https://doi.org/10.5285/2aa2c8ab-9e32-4b3b-9636-503912305aca) for more information on how these are calculated.

Simple examples are shown for exploring and working with the dataset, which can be generalised to any gridded dataset stored on S3 storage, a storage medium that is becoming increasingly common for large gridded datasets. Datasets on S3 storage are typically stored in the Zarr format. However if you are familiar with working with NetCDF files with Xarray, this will be largely transparent to you with respect to the commands you can use to view and analyse the dataset.

We will:

- Show how to open a dataset stored in Zarr format
- Show how to extract and plot a time series at a single location
- Show how to plot a map at a single time step
- Show how to extract out a catchment area for plotting

**Data store and chunking**

The data is currently publicly available as a trial through JASMIN object storage at the following URLs:
```
https://fdri-o.s3-ext.jc.rl.ac.uk/hydro-pe-spacechunk/hydro-pe_10km_chunks_all.zarr
https://fdri-o.s3-ext.jc.rl.ac.uk/hydro-pe-timechunk/hydro-pe_week_chunks.zarr
```
Each URL links to an identical copy of the dataset, but split up into chunks differently, to enable more performant access for particular types of analysis.

The 'hydro-pe-spacechunk' version is chunked across the spatial dimensions and is more performant for access patterns that extract out timeseries from a single gridpoint or small spatial areas.

The 'hydro-pe-timechunk' version is chunked across the time dimension and is more performant for access patterns that extract out large spatial areas at a single timestep or small number of timesteps.

---

## 0. Setup

Install and load the `stars` package and associated libraries. This can take several minutes.

In [ ]:
# ── Install packages ──────────────────────────────────────────────────────────
pkgs <- c("stars", "sf", "dplyr", "tidyr", "scales", "lubridate", "ggplot2", "CFtime")
new  <- pkgs[!pkgs %in% installed.packages()[, "Package"]]
if (length(new) > 0) install.packages(new, repos = "https://cloud.r-project.org", verbose = TRUE)

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
suppressPackageStartupMessages({
  library(stars)            # Spatiotemporal arrays / data cubes
  library(sf)               # Spatial features
  library(dplyr)            # Data wrangling
  library(tidyr)
  library(scales)           # Axis formatting
  library(lubridate)        # Date handling
  library(ggplot2)          # Plotting
})

cat("stars version:", as.character(packageVersion("stars")), "\n")

---
## 1. Open the Zarr store

Here, we read in all the dataset metadata using the store URL and gdal's Zarr support. We will read in both versions of the dataset.

Time-chunked version:

In [ ]:
# ── Store URL ─────────────────────────────────────────────────────────────────
store_url_timechunk <- "https://fdri-o.s3-ext.jc.rl.ac.uk/hydro-pe-timechunk/hydro-pe_week_chunks.zarr"

# Build the GDAL Zarr connection string
# read_mdim() uses GDAL's multidimensional array API, which handles time and
# all coordinate dimensions natively
gdal_url_timechunk <- paste0('ZARR:"/vsicurl/', store_url_timechunk, '"')

info_timechunk <- gdal_utils("mdiminfo", gdal_url_timechunk, quiet = TRUE)
info_json_timechunk <- jsonlite::fromJSON(info_timechunk, simplifyVector = FALSE)

print(info_json_timechunk)

Space-chunked version:

In [ ]:
# ── Store URL ─────────────────────────────────────────────────────────────────
store_url_spacechunk <- "https://fdri-o.s3-ext.jc.rl.ac.uk/hydro-pe-spacechunk/hydro-pe_10km_chunks_all.zarr"

# Build the GDAL Zarr connection string
# read_mdim() uses GDAL's multidimensional array API, which handles time and
# all coordinate dimensions natively
gdal_url_spacechunk <- paste0('ZARR:"/vsicurl/', store_url_spacechunk, '"')

info_spacechunk <- gdal_utils("mdiminfo", gdal_url_spacechunk, quiet = TRUE)
info_json_spacechunk <- jsonlite::fromJSON(info_spacechunk, simplifyVector = FALSE)

print(info_json_spacechunk)

There's a lot of information here and it isn't very pretty, so let's focus on pulling out the bits of metadata that we need.

---
## 2. Get more specific variable information

### Chunk sizes

First, let's check the chunking of the two datasets to confirm they are different. We will be looking at the ```rainfall``` variable.

We use the '$' syntax to navigate through the metadata tree. Here we are pulling out the ```dimensions``` (names), ```dimension_size```, and ```block_size``` information from the ```rainfall``` ```array```. The ```block_size``` represents the chunk sizes of each dimension.

In [ ]:
info_json_spacechunk$arrays$pet$dimensions
info_json_spacechunk$arrays$pet$dimension_size
info_json_spacechunk$arrays$pet$block_size

By comparing the space-chunked version above to the time-chunked version below, we can see that (only) the chunk sizes (```block_size```s) are different.

In the space-chunked version above we can see that the ```x``` and ```y``` dimensions are chunked into blocks of 10.

In the time-chunked version below we can that the ```time``` dimension is chunked into blocks of 7.

In each case, the other dimensions have a chunk size equal to or exceeding their dimension size - i.e. they are not split up or chunked.

In [ ]:
info_json_timechunk$arrays$pet$dimensions
info_json_timechunk$arrays$pet$dimension_size
info_json_timechunk$arrays$pet$block_size

### Attributes

Let's now look at some of the other metadata about the ```rainfall``` variable in the dataset. The metadata will be the same between the version of the dataset, so we can use either here.

In [ ]:
# Navigate to the variable's attributes
var_attrs <- info_json_spacechunk$arrays$pet$attributes
print(var_attrs)

### Variable dimensions


We can also use `stars`' built-in multi-dimension array reader to view some of the information. It can't access the full metadata like ```gdal``` can above, but can provide information on the dimensions of variables from the dataset.

In [ ]:
dataset_spacechunk <- read_mdim(gdal_url_spacechunk, variable = "pet", proxy=TRUE)

# ── Show dataset dimensions and basic metadata ───────────────────────────────
dims_spacechunk <- st_dimensions(dataset_spacechunk)
print(dims_spacechunk)

### Read coordinate arrays

We can also `stars`' dimension reader to see the coordinate values:

In [ ]:
# ── Extract coordinate vectors from st_dimensions() ─────────────────────────
# stars stores coordinate values directly in the dimension metadata;
# no need to read them as separate arrays

dims <- st_dimensions(dataset_spacechunk)

xs      <- st_get_dimension_values(dataset_spacechunk, "x")
ys    <- st_get_dimension_values(dataset_spacechunk, "y")
datetimes <- st_get_dimension_values(dataset_spacechunk, "time")

# stars decodes CF time automatically — datetimes is already a POSIXct vector
cat(sprintf("x   : %d values, %.3f to %.3f\n",
  length(xs), min(xs), max(xs)))
cat(sprintf("y   : %d values, %.3f to %.3f\n",
  length(ys), min(ys), max(ys)))
cat(sprintf("Time: %d steps, from %s to %s\n",
  length(datetimes), format(min(datetimes)), format(max(datetimes))))


Now that we've explored the metadata, we can get going with a couple of simple examples, using the appropriate chunking for each.

---
## 3. Time series at a single grid point

We will get better performance on this example if we use the version of the dataset that is chunked in space, and not in time. This store uses 10km × 10km chunks, therefore a single point time series will download only one chunk, the chunk that contains the gridpoint we've requested.

`stars` uses GDAL's `/vsicurl/` HTTP range request support under the hood, which ensures that only the chunks covering the requested subset are fetched from the cloud store.



In [ ]:
# ── Choose a target location ──────────────────────────────────────────────────
# E.g. Edinburgh, Scotland (British National Grid)
target_x <- 325000
target_y <- 673000

# Find the nearest grid cell coordinate values
nearest_x <- xs[which.min(abs(xs - target_x))]
nearest_y <- ys[which.min(abs(ys - target_y))]

cat(sprintf("Target location  : x=%.0f, y=%.0f\n", target_x, target_y))
cat(sprintf("Nearest grid cell: x=%.0f, y=%.0f\n", nearest_x, nearest_y))


Read a specific time period of the dataset for this location:

In [ ]:
# ── Read the time series for this grid cell using coordinate-based selection ──

# Specify the time range of interest
time_start <- as.POSIXct("2015-01-01", tz = "UTC")
time_end   <- as.POSIXct("2015-12-31 23:00:00", tz = "UTC")

# Find the time indices covering the requested range
t_idx <- which(datetimes >= time_start & datetimes <= time_end)
t_offset <- min(t_idx) - 1L
t_count  <- length(t_idx)

# Find the nearest grid cell indices
x_idx <- which.min(abs(xs - target_x))
y_idx <- which.min(abs(ys - target_y))

cat(sprintf("Reading %d time steps (%s to %s) for grid cell [x=%.0f, y=%.0f]...\n",
  t_count, format(time_start, "%Y-%m-%d"), format(time_end, "%Y-%m-%d"),
  xs[x_idx], ys[y_idx]))

# Read only the needed subset
ts_stars <- read_mdim(
  gdal_url_spacechunk, # <--- note using the space-chunked dataset version
  variable = "pet",
  offset   = c(x_idx - 1L, y_idx - 1L, t_offset),
  count    = c(1L, 1L, t_count)
)

# Build tidy data frame
ts_df <- data.frame(
  datetime = datetimes[t_idx],
  value    = as.numeric(ts_stars[[1]])
)
tail(ts_df)


Plot it

In [ ]:
# ── Plot: full time series ────────────────────────────────────────────────────

# Extract out needed metadata
var_long  <- "pet"
var_units <- info_json_spacechunk$arrays$pet$unit
y_label     <- if (nchar(var_units) > 0) paste0(var_long, "\n(", var_units, ")") else var_long
subtitle_ts <- sprintf("Grid cell: x=%.0f, y=%.0f (BNG EPSG:27700)", nearest_x, nearest_y)

# construct plot
p_ts <- ggplot(ts_df, aes(x = datetime, y = value)) + # data to plot
  geom_line(colour = "blue", linewidth = 0.5) + # line properties
  scale_x_datetime(labels = scales::date_format("%b %Y")) + # x axis label formatting
  scale_y_continuous(labels = scales::comma) + # y axis label formatting
  labs( # labels
    title    = "PET",
    subtitle = subtitle_ts,
    x        = NULL,
    y        = y_label,
    caption  = "Source: FDRI / UKCEH https://dx.doi.org/10.5285") +
  theme_minimal(base_size = 12)

print(p_ts)


---
## 4. Spatial map at a single time step

We will get better performance on this example if we use the version of the dataset that is chunked in time, and not in space. This store uses daily (24 hourly time step) chunks, therefore a spatial extraction at only one time step will download only one chunk, the chunk that contains the timestep we've requested.

As before, `stars` uses GDAL's `/vsicurl/` HTTP range request support under the hood, which ensures that only the chunks covering the requested subset are fetched from the cloud store. You are however likely to run into memory issues on Binder which is limited to 2GB of RAM. Try extracting out only a portion of the whole 2D slice if you encounter issues with memory.


In [ ]:
# ── Read the full spatial grid for a specific time step ───────────────────────

# Specify the time of interest
target_time <- as.POSIXct("2000-01-01", tz = "UTC")

# Find the nearest time step
t_idx  <- which.min(abs(as.POSIXct(datetimes) - target_time))
slice_time <- format(datetimes[t_idx], "%Y-%m-%d UTC")
cat(sprintf("Reading spatial slice for %s (time index %d)\n", slice_time, t_idx))

# Read the full x/y grid for that time step
slice_stars <- read_mdim(
  gdal_url_timechunk, # <--- note using the time-chunked dataset version
  variable = "pet",
  offset   = c(0L, 0L, t_idx - 1L),
  count    = c(length(xs), length(ys), 1L)
)

cat(sprintf("Grid: %d x × %d y\n", length(xs), length(ys)))
cat(sprintf("Value range: %.3f to %.3f %s\n",
  min(slice_stars[[1]], na.rm = TRUE),
  max(slice_stars[[1]], na.rm = TRUE),
  var_units))


To plot this nicely, several plotting libraries need to be installed. This can take upwards of 15 minutes. There are slight variations in how this needs to be done for different platforms. Follow the instructions and run the cells **only** for the platform you are currently running the notebook on.

#### Google Colab

In [ ]:
# Set up r2u for installing terra (uses precompiled binaries to bypass gdal problems)
download.file(
  "https://github.com/eddelbuettel/r2u/raw/master/inst/scripts/add_cranapt_focal.sh",
  "add_cranapt_focal.sh"
)
Sys.chmod("add_cranapt_focal.sh", "0755")
system("./add_cranapt_focal.sh")

#### JASMIN Notebook Service

JASMIN sometimes fails to find the proj directory, which contains the files needed for spatial plotting, meaning you have to set it manually. If you have followed the instructions for the JASMIN Notebook Service on the README, the proj directory should be installed in the directory written out in the commands below. You will need to replace "USERNAME" with your JASMIN username and "ENVNAME" with the name of the environment you created with conda when cloning the jasR environment (this is fdriR if you've followed the instructions word for word).

In [ ]:
# only needs to be run on the JASMIN Notebook Service.

proj_dir <- "/gws/ssde/j25b/fdri/envs/fdricombo/share/proj"
Sys.setenv(PROJ_DATA = file.path(proj_dir))
Sys.setenv(PROJ_LIB = file.path(proj_dir))

To check you have set this correctly, the following command should pass (i.e. not produce an error):

In [ ]:
stopifnot(file.exists(file.path(Sys.getenv("PROJ_DATA"), "proj.db")))

If it produces an error, it is likely proj_dir and thus PROJ_DATA is not correct. You can see what PROJ_DATA is with the below command. Double check proj_dir is set correctly.

In [ ]:
Sys.getenv("PROJ_DATA")

#### Binder

All packages are already set up and installed on Binder, no further configuration is required.

### All platforms - Install libraries

Packages will only be installed if they are not already.

In [ ]:
pkgs <- c("rnaturalearth", "rnaturalearthdata")
new  <- pkgs[!pkgs %in% installed.packages()[, "Package"]]
if (length(new) > 0) install.packages(new, repos = "https://cloud.r-project.org", verbose = TRUE)


In [ ]:
library('ggplot2')
library('rnaturalearth')
library('rnaturalearthdata')

### Plot

In [ ]:
# ── Plot the spatial slice using stars + ggplot2 ─────────────────────────────

# Drop the singleton time dimension so we have a clean 2D raster
slice_2d <- adrop(slice_stars)

# Set the EPSG code (the grid projection) For an OSGB grid it is 27700
st_crs(slice_2d) <- "EPSG:27700"

# Get UK coastline for overlay, transformed to the CRS
gb <- ne_countries(country = "United Kingdom",
                   scale = "medium", returnclass = "sf") |>
  st_transform(st_crs(slice_2d))

# geom_stars() renders a stars object directly in ggplot2
p_map <- ggplot() +
  geom_stars(data = slice_2d) + # render the data
  geom_sf(data = gb, fill = NA) + # render the coastlines
  scale_fill_distiller( # customise how the data is displayed
    palette   = "YlGnBu",
    direction = 1,
    name      = var_units,
    na.value  = NA,
    labels    = scales::comma
  ) +
  coord_sf(expand = FALSE) + # customise how the plot overall is displayed
  labs(
    title    = paste("PET"),
    subtitle = slice_time,
    x = "Lon", y = "Lat",
  ) +
  theme_minimal(base_size = 12)

print(p_map)


---
## Appendix

### Troubleshooting

| Symptom | Fix |
|---------|-----|
| `read_mdim()` errors with "no such file" | Check the GDAL Zarr URL format: must be `ZARR:"/vsicurl/https://..."` |
| Variable name not found | Run `gdal_utils("mdiminfo", gdal_url)` to see available variables; update the `variable =` argument |
| Map looks blank or all NA | Check `offset` and `count` values match the actual array dimension order |
| Time values look wrong | `stars` decodes CF time automatically; check `st_get_dimension_values(dataset, "time")` |
| Data is slow to extract | Expected over HTTP for large slices; GDAL fetches only the needed chunks but chunk size affects speed |
| Runtime crashes due to lack of RAM | Reading large slices can exhaust memory on Binder (2GB limit). Try a smaller subset or use a different platform |


---

**Data citation:** Hydro-PE, UKCEH, https://doi.org/10.5285/2aa2c8ab-9e32-4b3b-9636-503912305aca